# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [54]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [55]:
# TODO: Import the necessary libs
# For example: 
import os
from datetime import datetime
from typing import List

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage, BaseMessage
from lib.tooling import tool
from lib.parsers import PydanticOutputParser

import chromadb
from chromadb.utils import embedding_functions
from chromadb.api.models.Collection import Collection as ChromaCollection

from dotenv import load_dotenv

from pydantic import BaseModel, Field

from tavily import TavilyClient

import json

In [56]:
# TODO: Load environment variables

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [57]:
# TODO: Create retrieve_game tool
# It should use chroma client and collection you created

# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - query: a question about game industry. 
#
#    You'll receive results as list. Each element contains:
#    - Platform: like Game Boy, Playstation 5, Xbox 360...)
#    - Name: Name of the Game
#    - YearOfRelease: Year when that game was released for that platform
#    - Description: Additional details about the game

@tool
def retrieve_game(query:str):

    chroma_client = chromadb.PersistentClient(path="mychromadb")
    collection = chroma_client.get_collection("udaplay")

    results = collection.query(
        query_texts=[query],
        n_results=3,
        include=['documents']
    )

    retrieved_docs = results['documents'][0]

    return {"documents": retrieved_docs}

#### Evaluate Retrieval Tool

In [58]:
class RetrievalJudgeEvaluation(BaseModel):
    """Structured evaluation from LLM judge"""
    useful: bool = Field(description="Whether the documents are sufficient to answer the user question.")
    description: str = Field(description="Brief description about the evaluation result.")


# TODO: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

@tool
def evaluate_retrieval(question:str, documents):
    llm_judge = LLM(model="gpt-4o-mini")

    judge_prompt = f"""
    Your task is to evaluate if the documents are enough to respond the query.
    Give a detailed explanation, so it's possible to take an action to accept it or not.
    User question: {question}
    Retrieved documents: {documents}    
    """

    # Use structured output with Pydantic model
    judge_response = llm_judge.invoke(
        input=judge_prompt, 
        response_format=RetrievalJudgeEvaluation
    )

    # Parse the structured response
    parser = PydanticOutputParser(model_class=RetrievalJudgeEvaluation)
    try:
        evaluation_result = parser.parse(judge_response)
    except Exception as e:
        print(f"Debug: Structured parsing error: {e}")

    return evaluation_result

#### Game Web Search Tool

In [59]:
# TODO: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 

@tool
def game_web_search(question:str):
    
    api_key = os.getenv("TAVILY_API_KEY")
    client = TavilyClient(api_key=api_key)

    search_result = client.search(
        query=question,
        search_depth="advanced",
        include_answers=True,
        include_raw_content=False,
        include_images=False
    )

    # Format the results
    formatted_results = {
        "answer": search_result.get("answer", ""),
        "results": search_result.get("results", []),
        "search_metadata": {
            "timestamp": datetime.now().isoformat(),
            "query": question
        }
    }
    
    return formatted_results

### Agent

In [60]:
# TODO: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

tools=[retrieve_game, evaluate_retrieval, game_web_search]

udaplay_agent = Agent(
    model_name="gpt-4o-mini",
    instructions=(
            "You are a web-aware research agent for the video game industry. For each query, you will:"
            "Answer questions using internal knowledge (RAG)."
            "Evaluate if the internally retrieved knowledge is enough to answer the question correctly."
            "Search the web when needed using Tavily's AI-optimized search."
            "Maintain conversation state."
            "Return structured outputs that cite information sources, combine information from multiple" 
            "sources when needed and present information in a natural, readable format."
    ),
    tools=tools
)

In [61]:
# TODO: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

run1 = udaplay_agent.invoke(
    query="When Pokémon Gold and Silver was released?"
)

run2 = udaplay_agent.invoke(
    query="Which one was the first 3D platformer Mario game?"
)

run3 = udaplay_agent.invoke(
    query="Was Mortal Kombat X realeased for Playstation 5?"
)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor


In [62]:
def print_messages(messages: List[BaseMessage]):
    for m in messages: 
        print(f"(role = {m.role}, content = {m.content}, tool_calls = {getattr(m, 'tool_calls', None)})")

In [63]:
runs=[run1, run2, run3]

for i, run in enumerate(runs):
    run_messages=run.get_final_state()["messages"]
    print(f"Report for Run {i}:")
    print("Agents reasoning and tool usage:")
    print_messages(run_messages)
    print("Final answer:")
    print(run_messages[-1].content)
    print()

Report for Run 0:
Agents reasoning and tool usage:
(role = system, content = You are a web-aware research agent for the video game industry. For each query, you will:Answer questions using internal knowledge (RAG).Evaluate if the internally retrieved knowledge is enough to answer the question correctly.Search the web when needed using Tavily's AI-optimized search.Maintain conversation state.Return structured outputs that cite information sources, combine information from multiplesources when needed and present information in a natural, readable format., tool_calls = None)
(role = user, content = When Pokémon Gold and Silver was released?, tool_calls = None)
(role = assistant, content = None, tool_calls = [ChatCompletionMessageFunctionToolCall(id='call_jJjhgzSgHhSIHjsDZXBAWcak', function=Function(arguments='{"query":"Pokémon Gold and Silver release date"}', name='retrieve_game'), type='function')])
(role = tool, content = "{'documents': ['[Game Boy Color] Pok\u00e9mon Gold and Silver (1

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes